# 02 - Diffusion Training

本 notebook 演示 DDPM 在 VAE latent space 上的训练过程。

**前置条件**: 先运行 `01_vae_exploration.ipynb` 训练并保存 VAE。

In [ ]:
import sys
sys.path.append('..')

import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

from models.vae import VAE
from models.unet import UNet
from models.diffusion import GaussianDiffusion

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)

## 1. 加载预训练的 VAE

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

vae = VAE(latent_dim=1024).to(device)
vae.load_state_dict(torch.load('../checkpoints/vae.pt', map_location=device))
vae.eval()
print('VAE loaded successfully')

## 2. 查看 Latent Space 分布

In [ ]:
# 编码一批图像，查看 latent 分布
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])
dataset = datasets.MNIST(root='../data', train=True, download=True, transform=transform)
loader = DataLoader(dataset, batch_size=256, shuffle=True)

latents = []
with torch.no_grad():
    for x, _ in loader:
        z, _, _ = vae.encode(x.to(device))
        latents.append(z.cpu())
        if len(latents) > 10:  # 只取一部分
            break

latents = torch.cat(latents, dim=0)
print(f'Latent shape: {latents.shape}')
print(f'Latent mean: {latents.mean():.4f}, std: {latents.std():.4f}')

# 绘制分布
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].hist(latents.flatten().numpy(), bins=100, density=True, alpha=0.7)
axes[0].set_title('Latent Distribution')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Density')

# 随机选两个维度绘制散点图
axes[1].scatter(latents[:, 0].numpy(), latents[:, 1].numpy(), alpha=0.3, s=1)
axes[1].set_title('Latent Space (dim 0 vs dim 1)')
axes[1].set_xlabel('Dim 0')
axes[1].set_ylabel('Dim 1')
axes[1].set_aspect('equal')
plt.tight_layout()
plt.show()

## 3. 构建扩散模型

In [ ]:
unet = UNet(in_channels=64, base_ch=128).to(device)
diffusion = GaussianDiffusion(unet, timesteps=1000).to(device)

# 查看模型参数量
total_params = sum(p.numel() for p in unet.parameters())
print(f'U-Net parameters: {total_params:,}')

## 4. 可视化前向扩散过程

在训练之前，先看看前向加噪的过程是什么样的。

In [ ]:
# 取一张图像，逐步加噪
with torch.no_grad():
    x, _ = dataset[0]
    x = x.unsqueeze(0).to(device)
    z, _, _ = vae.encode(x)
    z = z / z.std()  # 归一化
    z = z.view(-1, 64, 4, 4)  # reshape for UNet
    
    # 不同时间步的加噪结果
    timesteps_to_show = [0, 100, 200, 500, 700, 999]
    noised_images = []
    for t_val in timesteps_to_show:
        t = torch.tensor([t_val], device=device)
        z_noised = diffusion.q_sample(z, t)
        # 解码回图像空间
        z_decoded = z_noised.view(-1, 1024) * z.std()
        img = vae.decode(z_decoded)
        noised_images.append(img.cpu())

fig, axes = plt.subplots(1, len(timesteps_to_show), figsize=(18, 3))
for i, (t_val, img) in enumerate(zip(timesteps_to_show, noised_images)):
    axes[i].imshow(img[0, 0] * 0.5 + 0.5, cmap='gray')
    axes[i].set_title(f't = {t_val}')
    axes[i].axis('off')
plt.suptitle('Forward Diffusion Process (Adding Noise)', fontsize=14)
plt.show()

## 5. 训练扩散模型

In [ ]:
optimizer = optim.Adam(unet.parameters(), lr=1e-3)
losses = []

num_epochs = 50
for epoch in range(num_epochs):
    diffusion.train()
    total_loss = 0
    pbar = tqdm(loader, desc=f'Epoch {epoch+1}/{num_epochs}')
    for x, _ in pbar:
        x = x.to(device)
        with torch.no_grad():
            z, _, _ = vae.encode(x)
            z = z / z.std()  # 归一化 latent
            z = z.view(-1, 64, 4, 4)  # reshape for UNet
        
        loss = diffusion.training_loss(z)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix(loss=loss.item())
    
    avg_loss = total_loss / len(loader)
    losses.append(avg_loss)
    print(f'Epoch {epoch+1}: Loss = {avg_loss:.4f}')

print('Training complete!')

## 6. 训练损失曲线

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE on noise)')
plt.title('Diffusion Training Loss')
plt.grid(True, alpha=0.3)
plt.show()

## 7. 生成样本

训练完成后，从纯噪声开始生成图像。

In [ ]:
diffusion.eval()
with torch.no_grad():
    # 从纯噪声采样
    z = diffusion.sample((16, 64, 4, 4), device)
    z = z.view(-1, 1024)  # reshape for VAE decoder
    generated = vae.decode(z)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(generated[i, 0].cpu() * 0.5 + 0.5, cmap='gray')
    ax.axis('off')
plt.suptitle('Generated Samples (from Noise)', fontsize=14)
plt.tight_layout()
plt.show()

## 8. 保存模型

In [ ]:
import os
os.makedirs('../checkpoints', exist_ok=True)
torch.save(unet.state_dict(), '../checkpoints/unet.pt')
print('U-Net saved to ../checkpoints/unet.pt')

## 总结

扩散模型学会了在 VAE 的 latent space 中逐步去噪，从随机噪声生成数字图像。

下一步: 运行 `03_sampling.ipynb` 探索更多采样技巧和可视化。